# Task 1 -- Define Source and Extract

## Source Definition

- **Name:** dev.to (Forem) public Articles API
- **Endpoint:** `GET https://dev.to/api/articles` (listing, filterable by `tag`) and `GET https://dev.to/api/articles/{id}` (single article, includes full body)
- **Authentication:** The listing endpoint is public - no API key required. An optional `DEVTO_API_KEY` (read from the environment, never hardcoded) is only needed for endpoints that act on your own account.
- **Rate limits:** dev.to rate-limits aggressive callers. This client retries on `HTTP 429` with exponential backoff.
- **Licence / terms of use:** Forem / dev.to Developer API -- https://developers.forem.com/api


In [1]:
from __future__ import annotations

import os
import time
from dataclasses import dataclass
from typing import Iterable, Iterator

import requests

DEVTO_API_BASE = "https://dev.to/api"


class DevToAPIError(RuntimeError):
    pass


@dataclass
class DevToClient:
    """Minimal client for the dev.to public API: pagination + retry/backoff on 429."""

    api_key: str | None = None
    base_url: str = DEVTO_API_BASE
    timeout: int = 15
    max_retries: int = 3
    backoff_seconds: float = 2.0

    def __post_init__(self):
        self.api_key = self.api_key or os.environ.get("DEVTO_API_KEY")
        self._session = requests.Session()
        headers = {"User-Agent": "edtech-content-pipeline/1.0"}
        if self.api_key:
            headers["api-key"] = self.api_key
        self._session.headers.update(headers)

    def _get(self, path: str, params: dict | None = None) -> list | dict:
        url = f"{self.base_url}{path}"
        last_exc = None
        for attempt in range(1, self.max_retries + 1):
            try:
                resp = self._session.get(url, params=params, timeout=self.timeout)
                if resp.status_code == 429:
                    time.sleep(self.backoff_seconds * attempt)
                    continue
                resp.raise_for_status()
                return resp.json()
            except requests.RequestException as exc:
                last_exc = exc
                time.sleep(self.backoff_seconds * attempt)
        raise DevToAPIError(f"GET {url} failed after {self.max_retries} attempts: {last_exc}")

    def get_articles(self, tag=None, page=1, per_page=30, top=None, username=None):
        """One page of articles."""
        params = {"page": page, "per_page": per_page}
        if tag:
            params["tag"] = tag
        if top:
            params["top"] = top
        if username:
            params["username"] = username
        data = self._get("/articles", params=params)
        return data if isinstance(data, list) else []

    def iter_articles_by_tag(self, tag, max_pages=5, per_page=30, top=None, sleep_between_pages=0.5):
        """Yields raw article dicts across multiple pages for a single tag."""
        for page in range(1, max_pages + 1):
            batch = self.get_articles(tag=tag, page=page, per_page=per_page, top=top)
            if not batch:
                break
            yield from batch
            if len(batch) < per_page:
                break
            time.sleep(sleep_between_pages)

    def get_article_by_id(self, article_id):
        """Full article, including body_markdown / body_html (listing endpoint never returns these)."""
        data = self._get(f"/articles/{article_id}")
        return data if isinstance(data, dict) else {}

    def get_articles_by_tags(self, tags, max_pages_per_tag=5, per_page=30, top=None):
        """Fetches and merges articles across several tags (raw, not deduped)."""
        results = []
        for tag in tags:
            results.extend(
                self.iter_articles_by_tag(tag=tag, max_pages=max_pages_per_tag, per_page=per_page, top=top)
            )
        return results


## Extract: fetch pages (with pagination + error handling)

In [2]:
TAGS = ["machinelearning", "datascience", "cloud", "aws", "ai"]
MAX_PAGES_PER_TAG = 2
PER_PAGE = 30

client = DevToClient()

print(f"Fetching dev.to articles for tags={TAGS} (up to {MAX_PAGES_PER_TAG} pages each)...")
raw_articles = client.get_articles_by_tags(tags=TAGS, max_pages_per_tag=MAX_PAGES_PER_TAG, per_page=PER_PAGE)
print(f"-> {len(raw_articles)} raw articles fetched (listing only, no body yet)")


Fetching dev.to articles for tags=['machinelearning', 'datascience', 'cloud', 'aws', 'ai'] (up to 2 pages each)...


-> 300 raw articles fetched (listing only, no body yet)


### Fetch full article content

The listing endpoint never returns the article body -- only `GET /articles/{id}` does. This
does one extra API call per article and merges `body_markdown` / `body_html` straight into
the same raw dict. Nothing is cleaned or transformed here -- the JSON saved next is still a
genuine raw API response, just enriched with a second raw endpoint's data. Text cleanup of
`body_markdown` (stripping Liquid embeds, code fences, HTML) happens in `02_profile_clean.ipynb`.

In [3]:
import time

CONTENT_DELAY = 0.3  # seconds between per-article content requests -- be polite to the API

total = len(raw_articles)
for i, article in enumerate(raw_articles, start=1):
    article_id = article.get("id")
    print(f"  [{i}/{total}] fetching full content for article {article_id}...", end="\r")
    try:
        detail = client.get_article_by_id(article_id)
        article["body_markdown"] = detail.get("body_markdown")
        article["body_html"] = detail.get("body_html")
    except Exception as exc:  # noqa: BLE001 - keep going even if one article fails
        print(f"\n  ! could not fetch content for article {article_id}: {exc}")
    time.sleep(CONTENT_DELAY)

with_content = sum(1 for a in raw_articles if a.get("body_markdown"))
print(f"\nDone -- {with_content}/{total} articles have body content")



Done -- 300/300 articles have body content


### Save the raw response, unmodified

In [4]:
import json
import os
from datetime import datetime, timezone

RAW_DIR = os.path.join("..", "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

# Task 1 deliverable: the raw API response (listing fields + body_markdown/body_html),
# saved as-is -- no cleaning or standardizing here.
timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d")
raw_path = os.path.join(RAW_DIR, f"devto_{timestamp}.json")
with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(raw_articles, f, ensure_ascii=False, indent=2)

print(f"Saved raw response -> {raw_path}")


Saved raw response -> ../data/raw/devto_2026-09-12.json


## Sanity check (preview only -- no files written here)

Profiling, flattening, and cleaning (including stripping HTML/Liquid noise out of
`body_markdown`) happen in `02_profile_clean.ipynb`, which reads straight from `data/raw/`.
This cell only previews the first raw record so we can confirm the extract worked.

In [5]:
import json

print(f"Total raw articles fetched this run: {len(raw_articles)}")
if raw_articles:
    preview = {k: v for k, v in raw_articles[0].items() if k != "body_html"}
    print(json.dumps(preview, indent=2)[:1200])


Total raw articles fetched this run: 300
{
  "type_of": "article",
  "id": 4630772,
  "title": "Most AI \"Reasoning\" Traces Are Just the Answer, Written Backwards",
  "description": "You've probably watched an AI think through a problem step by step, nod along with the logic, and...",
  "readable_publish_date": "Sep 11",
  "slug": "most-ai-reasoning-traces-are-just-the-answer-written-backwards-cho",
  "path": "/dj29/most-ai-reasoning-traces-are-just-the-answer-written-backwards-cho",
  "url": "https://dev.to/dj29/most-ai-reasoning-traces-are-just-the-answer-written-backwards-cho",
  "comments_count": 14,
  "public_reactions_count": 24,
  "collection_id": null,
  "published_timestamp": "2026-09-11T13:33:06Z",
  "language": "en",
  "subforem_id": 1,
  "ai_disclosure_level": "some_ai",
  "ai_disclosure_label": "AI-assisted",
  "positive_reactions_count": 24,
  "cover_image": "https://media2.dev.to/dynamic/image/width=1000,height=420,fit=cover,gravity=auto,format=auto/https%3A%2F%2Fdev-to